# P26 – Mental Health at Work
**Name:** RASHESH SHRIMANKAR
**Roll No.:** D34
**Enrollment No.:** 24004501210421
**Division:** D

**Problem:** Using the OSMI survey, predict whether a tech worker has sought mental-health treatment from age, gender, country, family history, work interference, company size, remote work and employer benefits.
**Dataset:** https://www.kaggle.com/datasets/osmi/mental-health-in-tech-survey
**Target (Y):** treatment (Yes/No)
**Model:** Decision Tree
**Main metric:** F1 and AUC

## Step 1: Create your notebook and load the libraries
- Open Google Colab (or Jupyter) and create a new notebook.
- Rename it as RollNo_P26_Name.ipynb (example: H35_P26_Riya.ipynb).
- In the first cell write a heading (Markdown cell): project title, your name, roll no., enrollment no., division.
- Run the code cell below. If you see 'All libraries loaded!' you are ready.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (confusion_matrix, ConfusionMatrixDisplay, classification_report,
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, RocCurveDisplay)
from sklearn.tree import DecisionTreeClassifier, plot_tree
import warnings; warnings.filterwarnings("ignore")
print("All libraries loaded!")

✍️ **Write here:** Title of the project and your details.

_(double-click this cell and write your answer)_

## Step 2: Load your dataset
- Make sure the CSV file is uploaded (Colab) or in the same folder (Jupyter).
- Run the code below. You should see the number of rows and columns, and the first 5 rows.

In [ ]:
df = pd.read_csv("survey.csv")
print(df.shape)      # (rows, columns)
df.head()

✍️ **Write here:** How many rows and columns does your dataset have? Where did it come from (Kaggle link)?

_(double-click this cell and write your answer)_

## Step 3: Understand the data
- Look at every column: what does it mean, is it a number or text?
- Check missing values and the target classes.
- Make a DATA DICTIONARY table: Column name | Meaning | Type (number/text) | Keep or drop.

In [ ]:
df.info()                   # column names, types, non-null counts
df.describe()               # min, max, mean of number columns
print(df.isnull().sum())    # missing values in each column
print("Duplicates:", df.duplicated().sum())

✍️ **Write here:** Data dictionary table + 2 lines about what you noticed.

_(double-click this cell and write your answer)_

## Step 4: Prepare the target column (Y)
- Your target (what the model predicts) is: treatment (Yes/No).
- The target already exists in the data. Just check how many rows each class has.
- If one class is much bigger than others, the data is IMBALANCED. Write this down: it affects which metric you trust.

In [ ]:
TARGET = "treatment"
print(df[TARGET].value_counts())                  # how many rows in each class
print(df[TARGET].value_counts(normalize=True))    # same, as percentages

✍️ **Write here:** Classes of the target and their counts. Is it balanced or imbalanced?

_(double-click this cell and write your answer)_

## Step 5: Clean the data
- Remove duplicate rows.
- Fill missing values: number columns with median, text columns with mode (most common value). Or drop a column if most of it is empty.
- Drop ID / name columns and any column that gives away the answer (leakage).
- Special note for YOUR dataset: Gender has 40+ messy spellings: clean them into a few groups. Age has impossible values (negative, 300+): treat as outliers. Drop Timestamp and comments. Add one paragraph on ethics: how should such a model be used, and not used?

In [ ]:
# 1) Remove duplicates
df = df.drop_duplicates()

# 2) Drop columns not needed: Timestamp, comments (text), state (mostly empty / US only), self_employed
#    and every column that is not one of the 8 inputs in the problem statement.
KEEP = ["Age", "Gender", "Country", "family_history", "work_interfere",
        "no_employees", "remote_work", "benefits", TARGET]
df = df[KEEP].copy()

# 3) Clean Gender (40+ spellings -> Male / Female / Other)
def clean_gender(g):
    g = str(g).strip().lower()
    male = {"male", "m", "man", "cis male", "cis man", "male (cis)", "mail", "maile", "make",
            "mal", "msle", "malr", "male-ish", "guy (-ish) ^_^", "something kinda male?", "ostensibly male, unsure what that really means"}
    female = {"female", "f", "woman", "cis female", "cis-female/femme", "female (cis)", "femake",
              "femail", "female ", "female (trans)", "trans-female", "trans woman"}
    if g in male: return "Male"
    if g in female: return "Female"
    return "Other"
df["Gender"] = df["Gender"].apply(clean_gender)
print(df["Gender"].value_counts())

# 4) Age outliers: keep realistic working ages 18-75, treat rest as missing then fill with median
df.loc[(df["Age"] < 18) | (df["Age"] > 75), "Age"] = np.nan

# 5) Group rare countries (<15 rows) into "Other" so one-hot encoding stays small
top = df["Country"].value_counts()
df["Country"] = df["Country"].where(df["Country"].map(top) >= 15, "Other")

# 6) Fill missing values: numbers -> median, text -> mode
for col in df.columns:
    if col == TARGET: continue
    if pd.api.types.is_numeric_dtype(df[col]):
        df[col] = df[col].fillna(df[col].median())
    else:
        df[col] = df[col].fillna(df[col].mode()[0])
df = df.dropna(subset=[TARGET])

print(df.isnull().sum().sum(), "missing values left")
print(df.shape)

✍️ **Write here:** What you removed or filled, and WHY (one line each).

_(double-click this cell and write your answer)_

## Step 6: Explore the data with charts (EDA)
- Run the charts below. Look at each chart carefully.
- Write 5 INSIGHTS in simple English. Example: 'Players with more gold at 10 minutes win more often.'

In [ ]:
sns.countplot(x=df[TARGET]); plt.title("Class balance"); plt.show()

num_cols = df.select_dtypes(include="number").columns
df[num_cols].hist(figsize=(6, 4), bins=30); plt.tight_layout(); plt.show()

# Box plot of Age by class
sns.boxplot(x=df[TARGET], y=df["Age"]); plt.title("Age by treatment"); plt.show()

# Treatment rate by important categorical columns
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
for ax, col in zip(axes.ravel(), ["family_history", "work_interfere", "Gender", "no_employees", "remote_work", "benefits"]):
    sns.countplot(data=df, x=col, hue=TARGET, ax=ax); ax.set_title(f"{col} vs treatment")
    ax.tick_params(axis="x", rotation=30)
plt.tight_layout(); plt.show()

# Correlation heat map (numeric + encoded target)
tmp = df[num_cols].copy(); tmp["treatment_num"] = (df[TARGET] == "Yes").astype(int)
plt.figure(figsize=(5, 4))
sns.heatmap(tmp.corr(), annot=True, cmap="coolwarm"); plt.title("Correlation heat map"); plt.show()

✍️ **Write here:** 5 insights from the charts (one line each).

_(double-click this cell and write your answer)_

## Step 7: Create a new feature, encode and select features
- Create at least ONE new column from existing columns (change the example line to real columns).
- One-hot encoding turns text columns into 0/1 columns.
- SelectKBest (filter method) keeps the most useful columns. Write down which columns were selected.

In [ ]:
# 1) NEW FEATURE: ordinal "work_interfere" score (Never=0 ... Often=3) and a combined risk score
interfere_map = {"Never": 0, "Rarely": 1, "Sometimes": 2, "Often": 3}
df["interfere_score"] = df["work_interfere"].map(interfere_map)
df["risk_score"] = df["interfere_score"] + (df["family_history"] == "Yes").astype(int)   # work interference + family history

# 2) Separate X (inputs) and y (target)
X = df.drop(columns=[TARGET])
le_y = LabelEncoder()
y = le_y.fit_transform(df[TARGET])
print("Classes:", list(le_y.classes_))     # class 0 = No, class 1 = Yes

# 3) ENCODING: one-hot encoding
X = pd.get_dummies(X, drop_first=True)
print(X.shape)

# 4) FEATURE SELECTION (filter method)
from sklearn.feature_selection import SelectKBest, f_classif
k = min(15, X.shape[1])
selector = SelectKBest(f_classif, k=k).fit(X, y)
scores = pd.Series(selector.scores_, index=X.columns).sort_values(ascending=False)
print(scores.head(20))
X = X[scores.head(k).index]
feature_names = list(X.columns)
print("Selected:", feature_names)

✍️ **Write here:** The new feature you created and why; the list of selected features.

_(double-click this cell and write your answer)_

## Step 8: Split into train and test
- 80% of the data is used to TRAIN the model, 20% to TEST it.
- stratify=y keeps the same class ratio in train and test.
- Decision Tree does not need scaling.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)
print(X_train.shape, X_test.shape)

✍️ **Write here:** Size of train and test sets.

_(double-click this cell and write your answer)_

## Step 9: Train your model: Decision Tree
- The evaluate() function below trains a model and calculates ALL metrics: accuracy, precision, recall, F1, specificity and AUC.
- Run it once for the BASELINE (default settings). You will see the confusion matrix and classification report.
- Use only this model in your project: Decision Tree.

In [ ]:
results = []     # every setting you try is stored here

def evaluate(name, model, show=False):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    cm = confusion_matrix(y_test, y_pred)
    tn, fp, fn, tp = cm.ravel()
    spec = tn / (tn + fp)
    try:
        proba = model.predict_proba(X_test)
        auc = roc_auc_score(y_test, proba[:, 1])
    except Exception:
        auc = None
    row = {"Setting": name,
           "Accuracy": accuracy_score(y_test, y_pred),
           "Precision": precision_score(y_test, y_pred, average="binary", zero_division=0),
           "Recall": recall_score(y_test, y_pred, average="binary", zero_division=0),
           "F1": f1_score(y_test, y_pred, average="binary", zero_division=0),
           "Specificity": spec, "AUC": auc}
    if show:
        ConfusionMatrixDisplay(cm, display_labels=le_y.classes_).plot(cmap="Blues")
        plt.title(name); plt.show()
        print(classification_report(y_test, y_pred, target_names=[str(c) for c in le_y.classes_]))
        if auc is not None:
            RocCurveDisplay.from_predictions(y_test, proba[:, 1]); plt.title("ROC - " + name); plt.show()
    print(row)
    return row

# BASELINE: your model with default settings
model = DecisionTreeClassifier(random_state=42)
results.append(evaluate("Baseline", model, show=True))

✍️ **Write here:** Baseline results. Explain the confusion matrix in words (how many correct, how many wrong).

_(double-click this cell and write your answer)_

## Step 10: Try different settings (tuning)
- Train DecisionTreeClassifier with criterion='gini' and with criterion='entropy' and compare.
- Try max_depth = 2 to 10 and plot train vs test accuracy; pick the depth that does not overfit.
- Visualise the final tree with plot_tree() (max_depth 3–4 for readability) and name the root-node feature.
- Plot feature importances and explain the top 3.
- Every setting is added to the results list automatically.

In [ ]:
# A) gini vs entropy
for c in ["gini", "entropy"]:
    results.append(evaluate(f"criterion={c}", DecisionTreeClassifier(criterion=c, random_state=42)))

# B) max_depth: train vs test accuracy (find overfitting)
depths = range(2, 11); tr, te = [], []
for dpt in depths:
    t = DecisionTreeClassifier(max_depth=dpt, random_state=42).fit(X_train, y_train)
    tr.append(t.score(X_train, y_train)); te.append(t.score(X_test, y_test))
plt.plot(depths, tr, marker="o", label="train"); plt.plot(depths, te, marker="o", label="test")
plt.xlabel("max_depth"); plt.ylabel("accuracy"); plt.legend(); plt.show()
best_depth = list(depths)[int(np.argmax(te))]
print("Best depth:", best_depth)
results.append(evaluate(f"max_depth={best_depth}", DecisionTreeClassifier(max_depth=best_depth, random_state=42), show=True))

# C) Draw the tree (top levels only)
tree = DecisionTreeClassifier(max_depth=best_depth, random_state=42).fit(X_train, y_train)
plt.figure(figsize=(20, 8))
plot_tree(tree, max_depth=3, feature_names=[str(f) for f in feature_names],
          class_names=[str(c) for c in le_y.classes_], filled=True, fontsize=8)
plt.show()
print("Root node feature:", feature_names[tree.tree_.feature[0]])

# D) Feature importance
pd.Series(tree.feature_importances_, index=feature_names).sort_values().tail(10).plot(kind="barh")
plt.title("Feature importance"); plt.show()

✍️ **Write here:** What changed when you changed each setting? Which one was best?

_(double-click this cell and write your answer)_

## Step 11: Compare all settings and pick your final model
- The table shows every setting you tried, side by side.
- Your most important metric is F1 and AUC. Choose the setting with the best F1.
- Write your best setting inside final_model = ...( ) and run it.

In [ ]:
res = pd.DataFrame(results).round(3)
display(res)

res.set_index("Setting")[["Accuracy", "Precision", "Recall", "F1", "AUC"]].plot(kind="bar", figsize=(10, 4))
plt.title("Comparison of settings"); plt.ylim(0, 1); plt.xticks(rotation=45, ha="right"); plt.show()

best = res.sort_values("F1", ascending=False).iloc[0]
print("Best setting by F1:", best["Setting"])

# FINAL MODEL: built automatically from the best setting
bs = best["Setting"]
if bs.startswith("max_depth="):
    final_model = DecisionTreeClassifier(max_depth=int(bs.split("=")[1]), random_state=42)
elif bs.startswith("criterion="):
    final_model = DecisionTreeClassifier(criterion=bs.split("=")[1], random_state=42)
else:
    final_model = DecisionTreeClassifier(random_state=42)
evaluate("FINAL", final_model, show=True)

✍️ **Write here:** Comparison table + bar chart + 3 lines: why you chose this final setting (talk about F1 and AUC).

_(double-click this cell and write your answer)_

## Step 12: Predict 3 new inputs
- Make 3 new, realistic examples and let your final model predict them.
- Check: does the prediction make sense? Explain in one line each.

In [ ]:
# 3 NEW realistic inputs built from the selected feature columns (all 0/1 dummies except Age/scores)
new = pd.DataFrame(0, index=range(3), columns=X.columns, dtype=float)
def setv(i, **kw):
    for k_, v_ in kw.items():
        if k_ in new.columns: new.loc[i, k_] = v_
# Person 1: 45 yrs, family history, work often affected -> expect Yes
setv(0, Age=45, family_history_Yes=1, work_interfere_Often=1, interfere_score=3, risk_score=4)
# Person 2: 25 yrs, no family history, mental health never interferes -> expect No
setv(1, Age=25, interfere_score=0, risk_score=0)
# Person 3: 35 yrs, family history, sometimes interferes, has benefits -> likely Yes
setv(2, Age=35, family_history_Yes=1, work_interfere_Sometimes=1, interfere_score=2, risk_score=3, benefits_Yes=1)
display(new)
pred = final_model.predict(new)
print("Predictions:", le_y.inverse_transform(pred))

✍️ **Write here:** The 3 inputs, the predictions, and whether they make sense.

_(double-click this cell and write your answer)_

## Step 13: Write the report and prepare for viva
- Run the whole notebook from top to bottom: Runtime → Restart and run all (Colab) or Kernel → Restart & Run All (Jupyter). Fix any error.
- Make a 4–6 page PDF report (structure is in 'What to submit').
- Prepare the viva questions given below. You must be able to explain EVERY line of your code.

✍️ **Write here:** Conclusion: what your model can do, how good it is, and what you learned.

_(double-click this cell and write your answer)_